# NMF Topic Model — England k=15

**Corpus:** 3,931 articles post-cleaning (3,943 raw → cleaned by removing GOV.UK footer phrases, FFT newsletter boilerplate, and articles <200 chars)

**Sources:** SchoolsWeek (~69%), GOV.UK, FFT, EPI, Nuffield, FED

**Model:** NMF, k=15, max_features=3000, min_df=3, init=nndsvd, max_iter=1000

**Purpose:** One of four England model variants trained for the contestability demonstration. `nmf_eng_30` (production baseline), `nmf_eng_15`, `nmf_eng_5`, and `nmf_eng_30_nm` (no media) show how specification choices — topic count and source composition — change what the model finds. See `docs/internal/current/technical/contestability_example.md`.

**Key finding:** k=15 is the midpoint where source diversity emerges. FFT and EPI — invisible at k=5 — become analytically visible, with FFT leading two topics (absence 35%, pupil attainment 24%) and EPI co-contributing to three. The k=5 catch-all mega-topic (49.3%) splits into four distinct policy areas. Single-source topics drop from 60% (k=5) to 40% (k=15). k acts as a volume knob for minority sources — the specification choice determines not just what topics are visible but whose perspective is represented.


# 0. Imports 

In [1]:
import sys
sys.path.insert(0, "/workspaces/AM1_topic_modelling")

import logging
import pandas as pd
import numpy as np
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from model_pipeline.training.s02_cleaning import run_cleaning
from model_pipeline.training.s03_spacy_processing import run_spacy_processing
from model_pipeline.training.s04_vectorisation import run_vectorisation, build_vectorizer
from model_pipeline.training.s05_nmf_training import run_nmf_training, get_top_words_per_topic
from model_pipeline.training.s06_topic_allocation import TOPIC_NAMES

import logging
logging.getLogger("gensim").setLevel(logging.WARNING)

from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

INFO:model_pipeline.training.s06_topic_allocation:Loaded 5 topic names from llm_topic_review.json


# 1. Load England training data

In [2]:
preprocessed_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_preprocessed.parquet")

if preprocessed_path.exists():
    print("Loading preprocessed data (skipping cleaning + spaCy)...")
    df = pd.read_parquet(preprocessed_path)
    SKIP_PREPROCESSING = True
else:
    csv_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training.csv")
    df = pd.read_csv(csv_path)
    SKIP_PREPROCESSING = False

print(f"Loaded: {df.shape}")
print(f"Skip preprocessing: {SKIP_PREPROCESSING}")

Loading preprocessed data (skipping cleaning + spaCy)...
Loaded: (3943, 18)
Skip preprocessing: True


# 2. Prepare text column

The Supabase schema has `title` and `text` as separate columns. The pipeline expects a combined `text` column. Also rename `article_date` → `date` to match pipeline expectations.

In [3]:
if not SKIP_PREPROCESSING:
    # Combine title + text (same as s01_data_loader)
    df["text"] = df["title"].fillna("") + "\n\n" + df["text"].fillna("")
    df["date"] = pd.to_datetime(df["article_date"], errors="coerce")
    print(f"Text combined. Sample length: {df['text'].str.len().describe()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


## 3. Cleaning (s02)

In [4]:
if not SKIP_PREPROCESSING:
    df = run_cleaning(df)
    print(f"After cleaning: {df.shape}")
    print(f"Empty text_clean: {(df['text_clean'].str.len() == 0).sum()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


# 4. spaCy processing (s03)

This takes a few minutes on ~4k articles.

In [5]:
if not SKIP_PREPROCESSING:
    df = run_spacy_processing(df)
    print(f"After spaCy: {df.shape}")
    print(f"Empty text_final: {(df['text_final'].str.len() == 0).sum()}")
    print(f"Avg tokens per doc: {df['tokens_final'].apply(len).mean():.0f}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


In [6]:
if not SKIP_PREPROCESSING:
    df.to_parquet("/workspaces/AM1_topic_modelling/data/training/eng_training_preprocessed.parquet")
    print(f"Saved preprocessed data: {df.shape}")
else:
    print("Already loaded from parquet")

Already loaded from parquet


# 5. TF-IDF vectorisation (s04)

In [7]:
# Using same params as config.yaml: min_df=3, max_df=0.85, max_features=3000, ngram_range=(1,2)
vec_out = run_vectorisation(df)
print(f"TF-IDF matrix: {vec_out.X.shape}")
print(f"Vocabulary size: {len(vec_out.feature_names)}")
print(f"Sample features: {vec_out.feature_names[:20].tolist()}")

INFO:model_pipeline.training.s04_vectorisation:Step 04 (vectorisation): starting. Input shape=(3943, 18)
INFO:model_pipeline.training.s04_vectorisation:TF-IDF shape: (3943, 3000)
INFO:model_pipeline.training.s04_vectorisation:Vectorizer params: min_df=3 max_df=0.85 max_features=3000 ngram_range=(1, 2)
INFO:model_pipeline.training.s04_vectorisation:Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence pupil', 'absence school', 'absent', 'absent pupil', 'absentee', 'absolute', 'abuse', 'academic', 'academies', 'academisation', 'academy', 'academy academy', 'academy financial', 'academy freedom', 'academy funding']


TF-IDF matrix: (3943, 3000)
Vocabulary size: 3000
Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence pupil', 'absence school', 'absent', 'absent pupil', 'absentee', 'absolute', 'abuse', 'academic', 'academies', 'academisation', 'academy', 'academy academy', 'academy financial', 'academy freedom', 'academy funding']


# 6. Train NMF (k=15)

In [8]:
nmf_out = run_nmf_training(vec_out.X, n_topics=15, random_state=42, init="nndsvd", max_iter=1000)
print(f"W shape: {nmf_out.W.shape}")
print(f"H shape: {nmf_out.H.shape}")
print(f"Reconstruction error: {nmf_out.reconstruction_error:.6f}")


INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): starting. X shape=(3943, 3000)
INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): complete.
INFO:model_pipeline.training.s05_nmf_training:W shape=(3943, 15) | H shape=(15, 3000)
INFO:model_pipeline.training.s05_nmf_training:Reconstruction error: 55.292565


W shape: (3943, 15)
H shape: (15, 3000)
Reconstruction error: 55.292565


# 7. Topic stability across random seeds

In [9]:
seeds = [42, 123, 456, 789, 1024]
H_matrices = []

for seed in seeds:
    model = NMF(n_components=15, init="nndsvda", random_state=seed, max_iter=1000)
    model.fit(vec_out.X)
    H_matrices.append(model.components_)
    print(f"Seed {seed}: recon error = {model.reconstruction_err_:.4f}")

# Compare all pairs of H matrices
pair_scores = []
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        sim = cosine_similarity(H_matrices[i], H_matrices[j])
        # Best match for each topic in run i against run j
        best_matches = sim.max(axis=1).mean()
        pair_scores.append(best_matches)
        print(f"Seeds {seeds[i]} vs {seeds[j]}: avg best-match similarity = {best_matches:.4f}")

avg_stability = np.mean(pair_scores)
print(f"\nOverall topic stability: {avg_stability:.4f}")
print(f"Previous stability (old model): 0.94")
print(f"Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable")


Seed 42: recon error = 55.2926
Seed 123: recon error = 55.2926
Seed 456: recon error = 55.2926
Seed 789: recon error = 55.2926
Seed 1024: recon error = 55.2926
Seeds 42 vs 123: avg best-match similarity = 1.0000
Seeds 42 vs 456: avg best-match similarity = 1.0000
Seeds 42 vs 789: avg best-match similarity = 1.0000
Seeds 42 vs 1024: avg best-match similarity = 1.0000
Seeds 123 vs 456: avg best-match similarity = 1.0000
Seeds 123 vs 789: avg best-match similarity = 1.0000
Seeds 123 vs 1024: avg best-match similarity = 1.0000
Seeds 456 vs 789: avg best-match similarity = 1.0000
Seeds 456 vs 1024: avg best-match similarity = 1.0000
Seeds 789 vs 1024: avg best-match similarity = 1.0000

Overall topic stability: 1.0000
Previous stability (old model): 0.94
Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable


#### Topic stability at k=15: perfect 1.0 — identical to k=5. Deterministic init (nndsvda) converges to the same solution every time at this resolution. At k=30, stability drops to 0.94. Somewhere between k=15 and k=30, the factorisation becomes complex enough that multiple solutions exist. The stability threshold is itself a specification finding — it marks where the model transitions from having one obvious answer to having multiple plausible ones.


# 8. Inspect topics — top words per topic

Compare with previous topic names to see if they still hold.

In [10]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=100)

# Display stopwords — these don't affect the model, just make the top words easier to read
display_stop = {
    "school", "education", "pupil", "student", "teacher", "year", "new", "work",
    "time", "say", "make", "good", "need", "use", "know", "want", "come", "take",
    "people", "government", "report", "system", "support", "include", "provide",
    "number", "change", "part", "set", "high", "low", "level", "national", "local",
    "public", "service", "also", "would", "could", "one", "two", "first", "last",
    "week", "month", "day", "told", "said", "according", "cent", "per", "per cent",
    "child", "children", "young", "staff", "area", "programme", "policy",
    "guidance", "framework", "response", "statement", "proposal", "approach",
    "review", "update", "document", "detail", "section", "datum", "figure",
    "survey", "rate", "score", "point", "proportion", "percentage",
    "organisation", "department", "committee", "institute", "foundation",
    "summit", "voice", "stakeholder", "partnership", "engagement",
    "scheme", "initiative", "pilot", "introduce", "implement", "launch",
    "office", "official", "notification", "recipient", "correspondence",
    "cookie", "banner", "subscribe", "contact", "submit", "accessibility",
    "share", "print", "visit", "site", "experience", "article", "news", "blog",
    "interesting", "fact", "previous", "current", "date", "information",
    "different", "large", "place", "individual", "view", "analysis",
    "thing", "way", "job"
}

In [11]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=30)

for topic_id, words in enumerate(topics):
    print(f"\nTopic {topic_id}: {', '.join(words)}")


Topic 0: child, parent, family, health, support, care, child school, young, life, early, government, mental, people, service, mental health, social, young people, poverty, standard, attendance, labour, system, school child, need, country, club, breakfast, right, child young, free

Topic 1: academy, authority education, late information, education skill, school academy, late, local authority, authority, skill, local, information action, assurance, information, skill vocational, vocational training, management assurance, academy funding, academy financial, esfa, assurance school, training school, provider education, financial management, education provider, management, esfa late, agency academy, vocational, action education, skill agency

Topic 2: trust, academy, academy trust, mat, ceo, executive, chief, chief executive, school trust, trust school, improvement, director, staff, board, chain, role, governance, leader, multi academy, large, school improvement, team, financial, trustee, t

In [12]:
TOPIC_NAMES_15 = {
    0: "child_welfare_and_early_years",
    1: "academy_financial_oversight",
    2: "academy_trust_governance",
    3: "teacher_recruitment_and_retention",
    4: "ofsted_inspection_reform",
    5: "gcse_attainment_and_grades",
    6: "pupil_attainment_and_disadvantage",
    7: "dfe_intervention_notices",
    8: "send_and_council_deficits",
    9: "teacher_pay_and_strikes",
    10: "skills_and_apprenticeships",
    11: "exam_regulation_and_ofqual",
    12: "raac_building_crisis",
    13: "school_absence_and_attendance",
    14: "school_funding_and_meals",
}

df["dominant_topic"] = nmf_out.W.argmax(axis=1)
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_15)
df["dominant_topic_name"].value_counts()


dominant_topic_name
skills_and_apprenticeships           479
academy_trust_governance             412
teacher_recruitment_and_retention    411
pupil_attainment_and_disadvantage    354
ofsted_inspection_reform             350
school_funding_and_meals             298
send_and_council_deficits            263
child_welfare_and_early_years        260
teacher_pay_and_strikes              208
exam_regulation_and_ofqual           203
raac_building_crisis                 203
academy_financial_oversight          152
gcse_attainment_and_grades           130
school_absence_and_attendance        130
dfe_intervention_notices              90
Name: count, dtype: int64

#### k=15 is the midpoint between k=5's source-proxy collapse and k=30's full thematic resolution. The k=5 catch-all mega-topic (49.3% of corpus) splits into four distinct policy areas: child welfare, pupil attainment, SEND/council deficits, and school absence. Teacher pay splits from recruitment/retention. Six policy debates invisible at k=5 — GCSE grades, SEND, RAAC, absence, exam regulation, apprenticeships — now have their own topics. Academy oversight topics (1, 2) survive unchanged from k=5, suggesting robust structure. At k=30, further splits would separate SEND from council finance and GCSE subjects from disadvantage gaps. Stability remains 1.0 (deterministic init) — the transition to multiple plausible solutions happens somewhere between k=15 and k=30.


# 9. Explore a topic — top articles + source concentration

In [13]:
def explore_topic(topic_id, n=5):
    """Show top N articles for a given topic, ranked by topic weight."""
    W = nmf_out.W
    topic_weights = W[:, topic_id]
    top_idx = topic_weights.argsort()[::-1][:n]

    # Topic words
    words = topics[topic_id]
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:20]
    print(f"TOPIC {topic_id} ({TOPIC_NAMES_15[topic_id]}) — top words: {', '.join(filtered)}")
    print(f"{'='*120}\n")

    for rank, idx in enumerate(top_idx, 1):
        row = df.iloc[idx]
        weight = topic_weights[idx]
        title = row.get("title", "No title")
        source = row.get("source", "Unknown")
        date = str(row.get("article_date", ""))[:10]
        text = row.get("text_clean", row.get("text", ""))
        if isinstance(text, str) and len(text) > 500:
            text = text[:500] + "..."

        print(f"[{rank}] weight={weight:.4f} | {source} | {date}")
        print(f"    {title}")
        print(f"    {text}\n")

# Source concentration across all topics
print("Source concentration (top 50 articles per topic):")
print("=" * 80)
for t in range(nmf_out.nmf_model.n_components):
    top_idx = nmf_out.W[:, t].argsort()[::-1][:50]
    breakdown = df.iloc[top_idx]['source'].value_counts()
    pct = (breakdown / breakdown.sum() * 100).round(0).astype(int)
    summary = ", ".join(f"{src} {p}%" for src, p in pct.items())
    print(f"Topic {t:>2} ({TOPIC_NAMES_15[t]}): {summary}")

# Usage: explore_topic(0) for topic 0, explore_topic(3, n=10) for 10 articles


Source concentration (top 50 articles per topic):
Topic  0 (child_welfare_and_early_years): gov 52%, schoolsweek 36%, fed 4%, epi 4%, fft 4%
Topic  1 (academy_financial_oversight): gov 100%
Topic  2 (academy_trust_governance): schoolsweek 88%, gov 12%
Topic  3 (teacher_recruitment_and_retention): schoolsweek 86%, gov 10%, epi 4%
Topic  4 (ofsted_inspection_reform): schoolsweek 86%, gov 12%, fft 2%
Topic  5 (gcse_attainment_and_grades): schoolsweek 46%, fft 34%, epi 12%, gov 8%
Topic  6 (pupil_attainment_and_disadvantage): fft 62%, schoolsweek 28%, epi 6%, gov 4%
Topic  7 (dfe_intervention_notices): gov 100%
Topic  8 (send_and_council_deficits): schoolsweek 98%, gov 2%
Topic  9 (teacher_pay_and_strikes): schoolsweek 100%
Topic 10 (skills_and_apprenticeships): gov 80%, schoolsweek 8%, fed 6%, nuffield 6%
Topic 11 (exam_regulation_and_ofqual): gov 62%, schoolsweek 38%
Topic 12 (raac_building_crisis): schoolsweek 94%, epi 4%, gov 2%
Topic 13 (school_absence_and_attendance): fft 52%, school

#### Source concentration at k=15: the midpoint between k=5's source-proxy collapse and k=30's mixed sources. Six topics remain single-source monologues (Gov 100% or SchoolsWeek 94–100%). But k=15 is where FFT and EPI first become analytically visible — at k=5, neither exceeded 9% of any topic. At k=15, FFT leads two topics (pupil disadvantage 62%, absence 52%) and EPI co-contributes to three. The specification choice of k doesn't just change granularity — it determines which voices are audible. A policymaker using k=5 would never know FFT's data-led perspective on attainment gaps exists in the corpus. The child welfare catch-all (49% at k=5) is shrinking as topics split off and take SchoolsWeek articles with them, shifting the source mix as k increases.


# 10. LLM-suggested topic names

Send top words per topic to Claude for naming suggestions. Compare with existing names.

In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv(Path("/workspaces/AM1_topic_modelling/.env"))
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Build keyword lists (without display stopwords)
topic_keyword_lines = []
for i, words in enumerate(topics):
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:30]
    topic_keyword_lines.append(f"Topic {i}: {', '.join(filtered)}")

my_name_lines = []
for i in range(15):
    my_name_lines.append(f"Topic {i}: proposed name is '{TOPIC_NAMES_15.get(i, 'unknown')}'")

prompt = f"""You are helping label topics from an NMF topic model trained on UK education policy documents (2023-2025).
The corpus includes articles from government departments, think tanks, media outlets, and research organisations in England.
This model has k=15 topics.

STEP 1: For each topic below, suggest a name based ONLY on the keywords. Do not look ahead to Step 2.

{chr(10).join(topic_keyword_lines)}

For each topic return:
- suggested_name (2-4 words, snake_case)
- explanation (one sentence)

STEP 2: Now compare your suggestions against these human-proposed names for the same topics:

{chr(10).join(my_name_lines)}

For each topic, assess:
- proposed_name_assessment: "AGREE" (proposed name fits the keywords well), "CLOSE" (reasonable but could be more precise), or "DISAGREE" (proposed name doesn't capture the topic well)
- proposed_name_note: one sentence explaining why

Return ONLY a JSON list combining both steps:
[
  {{"topic": 0, "suggested_name": "name", "explanation": "why", "proposed_name_assessment": "AGREE", "proposed_name_note": "why"}},
  ...
]
No other text, no markdown, no code fences."""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{"role": "user", "content": prompt}],
)

llm_text = response.content[0].text

# Strip markdown code fences if present
cleaned = re.sub(r'^```(?:json)?\n?', '', llm_text.strip())
cleaned = re.sub(r'\n?```$', '', cleaned.strip())
llm_results = json.loads(cleaned)

# Summary table
print(f"{'Topic':>5}  {'My Name':<40}  {'Status':<8}  {'LLM Suggestion':<40}  Notes")
print("=" * 160)
for r in llm_results:
    i = r["topic"]
    mine = TOPIC_NAMES_15.get(i, "???")
    status = r.get('proposed_name_assessment', 'N/A') or 'N/A'
    suggestion = r.get('suggested_name', 'N/A') or 'N/A'
    note = r.get('proposed_name_note', 'N/A') or 'N/A'
    print(f"{i:>5}  {mine:<40}  {status:<8}  {suggestion:<40}  {note}")

# Detailed explanations
print("\n\nDETAILED EXPLANATIONS:")
print("=" * 80)
for r in llm_results:
    suggestion = r.get('suggested_name', 'N/A') or 'N/A'
    explanation = r.get('explanation', 'N/A') or 'N/A'
    status = r.get('proposed_name_assessment', 'N/A') or 'N/A'
    note = r.get('proposed_name_note', 'N/A') or 'N/A'
    print(f"\nTopic {r['topic']}: {suggestion}")
    print(f"  Why: {explanation}")
    print(f"  My name '{TOPIC_NAMES_15.get(r['topic'], '???')}': {status} — {note}")

# Save
with open("/workspaces/AM1_topic_modelling/data/evaluation_outputs/llm_topic_review_k15.json", "w") as f:
    json.dump(llm_results, f, indent=2)
print("\nSaved to data/evaluation_outputs/llm_topic_review_k15.json")


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Topic  My Name                                   Status    LLM Suggestion                            Notes
    0  child_welfare_and_early_years             AGREE     child_family_wellbeing                    The proposed name accurately captures both the child welfare focus and early years emphasis evident in the keywords.
    1  academy_financial_oversight               CLOSE     academy_oversight_management              While financial oversight is prominent, the topic also includes broader management assurance and vocational training elements.
    2  academy_trust_governance                  AGREE     academy_trust_leadership                  The proposed name perfectly captures the focus on academy trust structures and governance arrangements.
    3  teacher_recruitment_and_retention         AGREE     teacher_workforce_issues                  The proposed name accurately reflects the dual focus on recruiting new teachers and retaining existing ones.
    4  ofsted_inspection_reform 

#### LLM naming validation at k=15: 13/15 AGREE, 2/15 CLOSE. Naming difficulty scales with k — at k=5 (80% agree) topics are trivially obvious, at k=15 (87%) interpretive divergence starts appearing, at k=30 13/30 required relabelling. The two CLOSE cases are revealing: "governance" vs "leadership" (Topic 2) and "reform" vs "controversy" (Topic 4) — same keywords, different framings that would lead policymakers toward different interpretations. These ambiguities don't exist at k=5 but multiply at k=30. The naming layer becomes increasingly contestable as k increases.


# 11. Topic distribution — how many articles per dominant topic?

In [15]:
dominant_topics = nmf_out.W.argmax(axis=1)
dominant_weights = nmf_out.W.max(axis=1)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
topic_df = pd.DataFrame({
    "topic_num": topic_counts.index,
    "topic_name": [TOPIC_NAMES_15.get(i, "???") for i in topic_counts.index],
    "count": topic_counts.values,
    "pct": (topic_counts.values / len(dominant_topics) * 100).round(1),
})
print(topic_df.to_string(index=False))
print(f"\nDominant weight — min: {dominant_weights.min():.4f}, mean: {dominant_weights.mean():.4f}, max: {dominant_weights.max():.4f}")


 topic_num                        topic_name  count  pct
         0     child_welfare_and_early_years    260  6.6
         1       academy_financial_oversight    152  3.9
         2          academy_trust_governance    412 10.4
         3 teacher_recruitment_and_retention    411 10.4
         4          ofsted_inspection_reform    350  8.9
         5        gcse_attainment_and_grades    130  3.3
         6 pupil_attainment_and_disadvantage    354  9.0
         7          dfe_intervention_notices     90  2.3
         8         send_and_council_deficits    263  6.7
         9           teacher_pay_and_strikes    208  5.3
        10        skills_and_apprenticeships    479 12.1
        11        exam_regulation_and_ofqual    203  5.1
        12              raac_building_crisis    203  5.1
        13     school_absence_and_attendance    130  3.3
        14          school_funding_and_meals    298  7.6

Dominant weight — min: 0.0137, mean: 0.1216, max: 0.3741


#### Topic distribution at k=15: no mega-topic. Largest is skills/apprenticeships at 12.1% — compare to k=5's catch-all at 49.3%. Range is 2.3%–12.1%, much more even than k=5 (4.1%–49.3%) though not as tight as k=30 (1.3%–7.4%). Mean dominant weight 0.122 (up from k=5's 0.092, approaching k=30's 0.131) — the model is more confident with more topics to choose from. Max weight 0.374 (up from 0.303). The false hierarchy is gone and model uncertainty decreases as k increases. More topics doesn't just mean more granularity — it means more confident assignments per article.


# 12. Topic distribution by source

Check if any source dominates a topic (specification choice: corpus composition).

In [16]:
df["dominant_topic"] = dominant_topics
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_15)
ct = pd.crosstab(df["source"], df["dominant_topic_name"], normalize="columns").round(2)
print("Source distribution per topic (column-normalised):")
ct


Source distribution per topic (column-normalised):


dominant_topic_name,academy_financial_oversight,academy_trust_governance,child_welfare_and_early_years,dfe_intervention_notices,exam_regulation_and_ofqual,gcse_attainment_and_grades,ofsted_inspection_reform,pupil_attainment_and_disadvantage,raac_building_crisis,school_absence_and_attendance,school_funding_and_meals,send_and_council_deficits,skills_and_apprenticeships,teacher_pay_and_strikes,teacher_recruitment_and_retention
source,,,,,,,,,,,,,,,
epi,0.00,0.00,0.03,0.01,0.01,0.10,0.01,0.09,0.01,0.05,0.03,0.00,0.05,0.00,0.02
fed,0.01,0.02,0.02,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.16,0.00,0.01
fft,0.01,0.01,0.02,0.00,0.00,0.32,0.04,0.24,0.00,0.35,0.00,0.00,0.01,0.00,0.01
gov,0.99,0.06,0.28,0.90,0.35,0.08,0.13,0.05,0.06,0.07,0.07,0.02,0.26,0.03,0.06
nuffield,0.00,0.00,0.07,0.00,0.00,0.00,0.01,0.01,0.00,0.00,0.01,0.02,0.15,0.00,0.01
schoolsweek,0.00,0.91,0.58,0.09,0.63,0.49,0.82,0.61,0.93,0.54,0.89,0.95,0.37,0.96,0.89


#### Normalised source-topic heatmap at k=15: FFT becomes a real analytical voice — absence 35%, GCSE 32%, pupil attainment 24%. At k=5, FFT never exceeded 9%. EPI appears in attainment topics (10%, 9%). Nuffield (15%) and FED (16%) emerge in skills/apprenticeships — the most multi-source topic at any k. SchoolsWeek still dominates most topics but concentration is lower: at k=5 it was 61–92% of 4/5 topics, at k=15 it ranges 37–96% with more variation. Single-source topics drop from 60% (k=5) to 40% (k=15). k is a volume knob for minority sources — the specification choice determines not just what topics are visible but whose perspective is represented.


# 13. Save retrained model artifacts

In [17]:
import json
from datetime import datetime

run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
run_dir = Path(f"/workspaces/AM1_topic_modelling/experiments/outputs/runs/{run_id}")
run_dir.mkdir(parents=True, exist_ok=True)

# Topic summary for sensitivity page
topic_summary = {
    "model_id": "nmf_eng_15",
    "n_topics": 15,
    "n_articles": len(df),
    "topics": [
        {"topic_num": i, "name": TOPIC_NAMES_15[i], "count": int((nmf_out.W.argmax(axis=1) == i).sum())}
        for i in range(15)
    ]
}
with open(run_dir / "topic_summary.json", "w") as f:
    json.dump(topic_summary, f, indent=2)

# Topic names
with open(run_dir / "topic_names.json", "w") as f:
    json.dump(TOPIC_NAMES_15, f, indent=2)

# Run metadata
metadata = {
    "run_id": run_id,
    "model_id": "nmf_eng_15",
    "n_articles": len(df),
    "n_topics": 15,
    "init": "nndsvd",
    "random_state": 42,
    "max_iter": 1000,
    "reconstruction_error": float(nmf_out.reconstruction_error),
    "corpus": "eng_training_3943_supabase",
    "note": "England k=15 for contestability comparison. Not a production model."
}
with open(run_dir / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {run_dir}")


Saved to /workspaces/AM1_topic_modelling/experiments/outputs/runs/2026-03-24_112503


# 15. Save dashboard summary JSON (for Specification Sensitivity page)

In [18]:
import json

dominant = nmf_out.W.argmax(axis=1)

topic_data = []
for i in range(15):
    mask = dominant == i
    count = int(mask.sum())
    source_counts = df.loc[mask, "source"].value_counts()
    top_source = source_counts.index[0] if len(source_counts) > 0 else "unknown"
    top_source_pct = round(float(source_counts.iloc[0] / source_counts.sum()), 2) if len(source_counts) > 0 else 0

    topic_data.append({
        "topic_num": i,
        "name": TOPIC_NAMES_15[i],
        "count": count,
        "pct": round(count / len(df) * 100, 1),
        "top_source": top_source,
        "top_source_pct": top_source_pct,
        "single_source": top_source_pct > 0.90,
    })

summary = {
    "model_id": "nmf_eng_15",
    "n_topics": 15,
    "n_articles": len(df),
    "metrics": {
        "reconstruction_error": round(float(nmf_out.reconstruction_error), 4),
        "stability": round(float(avg_stability), 4),
        "mean_dominant_weight": round(float(nmf_out.W.max(axis=1).mean()), 4),
        "max_dominant_weight": round(float(nmf_out.W.max(axis=1).max()), 4),
    },
    "topics": topic_data,
}

out_path = "/workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_15_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved to {out_path}")
print(f"Single-source topics: {sum(1 for t in topic_data if t['single_source'])}/15")


Saved to /workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_15_summary.json
Single-source topics: 5/15


## Final Summary: England k=15 and Source Visibility

This notebook trained an NMF topic model on the England corpus (3,931 articles) at k=15 — one of four England model variants built to demonstrate how specification choices shape AI outputs. The same corpus, same preprocessing, same model architecture as the production k=30 model — only the number of topics changed.

**Fifteen topics recovered:** child_welfare_and_early_years, academy_financial_oversight, academy_trust_governance, teacher_recruitment_and_retention, ofsted_inspection_reform, gcse_attainment_and_grades, pupil_attainment_and_disadvantage, dfe_intervention_notices, send_and_council_deficits, teacher_pay_and_strikes, skills_and_apprenticeships, exam_regulation_and_ofqual, raac_building_crisis, school_absence_and_attendance, school_funding_and_meals. LLM validation agreed with 13/15 human-assigned names, with two rated CLOSE ("governance" vs "leadership", "reform" vs "controversy").

**k=15 is the midpoint in the specification sensitivity gradient.** The k=5 catch-all mega-topic (49.3% of corpus) splits into four distinct policy areas. Six policy debates invisible at k=5 — GCSE grades, SEND, RAAC, absence, exam regulation, apprenticeships — now have their own topics. Teacher pay separates from recruitment/retention. Academy oversight topics survive unchanged from k=5, suggesting robust structure.

**The core finding is that k=15 is where minority sources become audible.** FFT leads two topics (pupil attainment 24%, absence 35%) and EPI co-contributes to three. At k=5, neither exceeded 9% of any topic. Nuffield (15%) and FED (16%) emerge in skills/apprenticeships. Single-source topics drop from 60% at k=5 to 40% at k=15. The specification choice of k is a volume knob for minority sources — it determines whose perspective is represented in the analysis.

**Comparison across k reveals a clear progression:**
1. **Source diversity** — k=5 (3/5 single-source) → k=15 (6/15 single-source) → k=30 (fewer). Minority voices emerge as k increases.
2. **Topic distribution** — k=5 (one topic at 49%) → k=15 (largest at 12.1%) → k=30 (largest at 7.4%). False hierarchy disappears.
3. **Model confidence** — k=5 (mean weight 0.092) → k=15 (0.122) → k=30 (0.131). More topics = more confident assignments.
4. **Naming contestability** — k=5 (80% LLM agree) → k=15 (87%) → k=30 (13/30 relabelled). Naming gets harder as topics get more specific.
5. **Stability** — k=5 (1.0) → k=15 (1.0) → k=30 (0.94). The transition to multiple plausible solutions happens between k=15 and k=30.

**All five metrics tell the same story:** k is not a technical parameter. It determines what the model measures, whose voice is audible, how confident the assignments are, how contestable the naming is, and whether the solution is unique. A policymaker would reach different conclusions at each k — and standard quality metrics won't flag which is "right."

This model is not intended for production use. Its purpose is to provide pre-computed results for the Specification Sensitivity dashboard page, where toggling between k=5, k=15, and k=30 makes the abstract argument about specification sensitivity concrete and interactive. See `docs/internal/current/technical/contestability_example.md` for the full contestability analysis.
